# Laboratório: Classificador de URLs Maliciosas em Larga Escala
Este laboratório utiliza o Google Colab para criar um classificador de URLs maliciosas usando um dataset de 1 000 000 de domínios benignos (Cisco Umbrella) e o feed do PhishTank.

**Objetivos:**
1. Carregar e rotular datasets grandes de URLs.
2. Pré-processar URLs com HashingVectorizer (vetor esparso).
3. Treinar modelo online (SGDClassifier) em minibatches.
4. Avaliar desempenho.

---

## 1. Instalação e imports

In [1]:
!pip install scikit-learn pandas wget --quiet

import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import wget
import zipfile

print('Dependências carregadas')

  Preparing metadata (setup.py) ... done
Dependências carregadas


## 2. Download e preparação dos dados

In [2]:
# 2. abrir e processar feeds de URLs
import io
import requests
import wget
import zipfile
import pandas as pd

# 2.1. Feed PhishTank (maliciosas) – usando o .csv.gz direto para evitar erros de ZIP
phish_url_gz = 'https://data.phishtank.com/data/online-valid.csv.gz'
headers = {'User-Agent': 'phishtank/seu_email@example.com'}  # PhishTank exige User-Agent descritivo
resp = requests.get(phish_url_gz, headers=headers)
resp.raise_for_status()
# pandas consegue ler gzip em memória
phish = pd.read_csv(io.BytesIO(resp.content), compression='gzip', usecols=['url'])
phish['label'] = 1  # 1 = maliciosa

# 2.2. Cisco Umbrella Top-1M (benignas)
umbrella_zip = 'umbrella_top1m.zip'
wget.download('https://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip', umbrella_zip)
with zipfile.ZipFile(umbrella_zip, 'r') as z:
    z.extractall()  # gera 'top-1m.csv'
umbrella = pd.read_csv('top-1m.csv', names=['rank','url'])
umbrella = umbrella[['url']].copy()
umbrella['label'] = 0  # 0 = benign

# 2.3. Majestic Million (benignas)
wget.download('https://downloads.majestic.com/majestic_million.csv', 'majestic_million.csv')
majestic = pd.read_csv('majestic_million.csv', skiprows=1, names=['rank','url','ref_domains'])
majestic = majestic[['url']].copy()
majestic['label'] = 0

# 2.4. Unir listas benignas e remover duplicatas
benign_all = pd.concat([umbrella, majestic], ignore_index=True)
benign_all = benign_all.drop_duplicates(subset='url').reset_index(drop=True)

# 2.5. Balanceamento: 2 benignas para cada 1 maliciosa
n_mal = len(phish)
n_ben = 2 * n_mal
benign = benign_all.sample(n=n_ben, random_state=42).reset_index(drop=True)

# 2.6. Combinar e embaralhar todo o dataset
data = pd.concat([phish, benign], ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total de URLs: {len(data)}")
print(data['label'].value_counts())


Total de URLs: 184506
label
0    123004
1     61502
Name: count, dtype: int64


## 3. Pré-processamento com HashingVectorizer

In [3]:
# 3. Pré-processamento com HashingVectorizer

# 3.1. Garanta que toda URL seja str
data['url'] = data['url'].astype(str)

# 3.2. Divisão em treino e teste (já com strings)
from sklearn.model_selection import train_test_split
urls_train, urls_test, y_train, y_test = train_test_split(
    data['url'], data['label'].values,
    test_size=0.2, random_state=42
)

# 3.3. Instanciar o vectorizer
from sklearn.feature_extraction.text import HashingVectorizer
vectorizer = HashingVectorizer(analyzer='char', ngram_range=(3,5), n_features=5000)

# 3.4. (Opcional) Veja a forma esparsa de um exemplo
print(vectorizer.transform([urls_train.iloc[0]]).shape)  # deve mostrar (1, 5000)

print("Pré-processamento pronto — URLs convertidas em vetores esparsos.")


(1, 5000)
Pré-processamento pronto — URLs convertidas em vetores esparsos.


## 4. Treinamento online com SGDClassifier e Avaliação

In [ ]:
# 1. Dividir URLs antes de vetorização
urls = data['url'].astype(str)           # força todas as URLs a str
labels = data['label'].values
urls_train, urls_test, y_train, y_test = train_test_split(
    urls, labels, test_size=0.2, random_state=42
)

# 2. Vetorizador de hashing (só uma vez)
vectorizer = HashingVectorizer(analyzer='char', ngram_range=(3,5), n_features=5000)

# 3. Inicializar o classificador
clf = SGDClassifier(loss='log_loss', max_iter=1, warm_start=True)

# 4. Primeiro partial_fit para criar internamente os coeficientes
X_init = vectorizer.transform(urls_train.iloc[0:1000])
y_init = y_train[0:1000]
clf.partial_fit(X_init, y_init, classes=[0,1])

# 5. Loop em batches maiores
batch_size = 100_000
for start in range(1000, len(urls_train), batch_size):
    end = start + batch_size
    batch_urls = urls_train.iloc[start:end]
    X_batch = vectorizer.transform(batch_urls)
    y_batch = y_train[start:start + batch_urls.shape[0]]
    clf.partial_fit(X_batch, y_batch)
    print(f"Treinado batch {start}–{start + batch_urls.shape[0]}")

# 6. Avaliação
X_test = vectorizer.transform(urls_test)
y_pred = clf.predict(X_test)
print("Acurácia:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Treinado batch 1000–101000


## 5. Mostrar exemplos das classificações

In [ ]:
import pandas as pd

# 1. Transformar as URLs de teste em features
X_test = vectorizer.transform(urls_test)

# 2. Obter previsões
y_pred = clf.predict(X_test)

# 3. Construir DataFrame com resultados
results = pd.DataFrame({
    'url': urls_test.values,
    'true_label': y_test,
    'pred_label': y_pred
})

# 4. Filtrar e mostrar algumas amostras
print("Exemplos de URLs classificadas como BENIGNAS (pred_label = 0):")
display(results[results.pred_label == 0].head(10))

print("\nExemplos de URLs classificadas como MALICIOSAS (pred_label = 1):")
display(results[results.pred_label == 1].head(10))


Exemplos de URLs classificadas como BENIGNAS (pred_label = 0):


,url,true_label,pred_label
0,dccon.dcinside.com,0,0
1,larian.com,0,0
3,0.0.0.0.0.1.1.c.0.c.0.0.4.0.b.8.f.7.0.6.2.ip6....,0,0
5,actionkit.com,0,0
6,founddevice.com,0,0
7,pull-lls-h6.douyincdn.com,0,0
8,72.172.35.in-addr.arpa,0,0
10,v4-kling.kechuangai.com,0,0
11,nflflag.com,0,0
13,affluhub888.com,0,0



Exemplos de URLs classificadas como MALICIOSAS (pred_label = 1):


,url,true_label,pred_label
2,https://bit.ly/3CygwTJ,1,1
4,https://qrco.de/bfeYdf,1,1
9,https://bit.ly/4bD2tZc,1,1
12,https://pub-9b5d9eb5e0f34c829334eacd67387487.r...,1,1
17,https://bit.ly/3Nv8FIQ,1,1
19,https://q-r.to/bfK3pO,1,1
21,https://docs.google.com/presentation/d/e/2PACX...,1,1
24,https://sites.google.com/view/jkhh8/home,1,1
25,https://q-r.to/bfIKO0,1,1
26,https://dustinkarper231.wixstudio.io/home,1,1



## 6. Análise de URLs avulsas

In [ ]:
# 6. testar URLs avulsas em loop até o usuário encerrar
def prever_url(url, vectorizer, model):
    """
    Recebe uma URL (string), aplica vetorização e retorna 'MALICIOSA' ou 'BENIGNA'.
    """
    X = vectorizer.transform([url])
    pred = model.predict(X)[0]
    return 'MALICIOSA' if pred == 1 else 'BENIGNA'

if __name__ == '__main__':
    while True:
        url_teste = input("Digite uma URL para classificação (ou 'sair' para encerrar): ").strip()
        if url_teste.lower() in ('sair', 'exit', 'quit'):
            print("Encerrando o script. Até mais!")
            break

        resultado = prever_url(url_teste, vectorizer, clf)
        print(f"A URL '{url_teste}' foi classificada como: {resultado}")

        # perguntando se quer continuar (opcional, pois já se pode usar 'sair' acima)
        cont = input("Deseja classificar outra URL? (s/n): ").strip().lower()
        if cont not in ('s', 'sim', 'y', 'yes'):
            print("Encerrando o script. Até mais!")
            break


Digite uma URL para classificação (ou 'sair' para encerrar): www.ipog.edu.br
A URL 'www.ipog.edu.br' foi classificada como: BENIGNA
Deseja classificar outra URL? (s/n): s
Digite uma URL para classificação (ou 'sair' para encerrar): https://www.exploit-db.com/
A URL 'https://www.exploit-db.com/' foi classificada como: MALICIOSA
Deseja classificar outra URL? (s/n): s
Digite uma URL para classificação (ou 'sair' para encerrar): https://gatry.com/
A URL 'https://gatry.com/' foi classificada como: MALICIOSA
Deseja classificar outra URL? (s/n): n
Encerrando o script. Até mais!
